# 1. Introduction to EDA

### Exploratory Data Analysis

This study focuses on detecting Distributed Denial-of-Service (DDoS) attacks using an unsupervised deep learning approach based on autoencoders. The CIC-DDoS2019 dataset is employed due to its diversity of modern attack scenarios and its flow-based feature representation. Prior to model development, an exploratory data analysis (EDA) is conducted to understand feature semantics, data quality, and statistical properties.

### Why CIC-DDoS2019 is used

This dataset contains recent network traffic data with binary labeling, distinguishing between benign traffic and DDoS attacks. In this configuration, all DDoS attack types are grouped under a single “DDoS” label to simplify the classification task and avoid ambiguity between specific attack variants.

The dataset is derived from real-world packet capture (PCAP) traffic and processed using CICFlowMeter-V3, which extracts flow-based statistical features from raw network packets. As a result, each record represents a labeled network flow rather than raw packet data.


License:
```
"DDoS Evaluation Dataset (CIC-DDoS2019).” DDoS 2019 | Datasets | Research | Canadian Insitute for Cybersecurity, 31 Oct. 2019, www.unb.ca/cic/datasets/ddos-2019.html.

Iman Sharafaldin, Arash Habibi Lashkari, Saqib Hakak, and Ali A. Ghorbani, "Developing Realistic Distributed Denial of Service (DDoS) Attack Dataset and Taxonomy", IEEE 53rd International Carnahan Conference on Security Technology, Chennai, India, 2019.
```

### Why unsupervised learning (autoencoder)

Autoencoders are well-suited for anomaly detection tasks because they can be trained primarily on unlabeled or normal traffic data. This reduces the dependency on large-scale labeled datasets, which are often costly and difficult to obtain in cybersecurity contexts.

Furthermore, once trained on benign behavior, the model learns a compressed representation of normal traffic patterns. Anomalies can then be identified based on reconstruction error using a predefined threshold, enabling the detection of previously unseen attack patterns.

# 2. Dataset Overview

In [ ]:
import pandas as pd
import numpy as np
from IPython.display import display

df = pd.read_csv("./data/raw/CIC-DDos2019.csv")

# Several column have a blank space before the actual name
df.columns = df.columns.str.strip()
df.columns = df.columns.str.replace(" ", "_")
df.columns = df.columns.str.lower()

display(df['flow_id'].nunique())
display(df.columns)
display(df.dtypes)

### Overall Structure

The dataset contains more than 225,000 rows, where each row represents a *network flow*. A flow is defined as a sequence of packets exchanged between two endpoints, which is aggregated using CICFlowMeter. The dataset is structured into 85 columns; however, not all features are equally informative. Therefore, the next step of the analysis focuses on identifying which features or groups of features are most relevant for model learning.

To achieve this, the features will be organized into meaningful categories, allowing for a more structured evaluation of their importance.

First, we categorize the features into distinct groups. Each group represents a different aspect of the network traffic that the autoencoder can learn from: *behavioral features*, which describe how a flow behaves; *identity features*, which characterize the communicating hosts; and *contextual features*, which provide information about the connection environment.

The objective of the model is to learn the typical behavior of benign network flows in order to identify those associated with malicious DDoS activity. To achieve this, *behavioral features* are expected to provide the most relevant information. *Some contextual features* may also be useful, as they help explain traffic patterns according to the nature of the connection. For example, a rate of 500 packets per second may be normal for a DNS flow, whereas it would be unusual for an SSH connection. Finally, *identity features* will be excluded from the analysis. Such features may introduce dataset-specific biases and encourage overfitting, since the model could rely on recurring identifiers (e.g., IP addresses, ports, and flow IDs) rather than learning the underlying behavioral patterns that distinguish benign traffic from DDoS attacks.

In [ ]:
#We select the columns that correspond to identity features
identity_features = ['flow_id', 'source_ip', 'source_port', 'destination_ip', 'destination_port']

try:
    df = df.drop(columns=identity_features)
except:
    print("Columns already dropped")

display(df.columns)
display(df.shape)

### Near-constant features

Features with near-constant values provide little to no information gain, as they exhibit extremely low variance across the dataset. In an unsupervised setting such as autoencoders, these features can introduce noise in the optimization process without contributing meaningful structure for reconstruction or anomaly detection.
These datasets often carry infinite values, in order to that we will remove this by, first computing infinite values as NaN, and then droping every null value.

In [ ]:
#Features with less-equal 1 different value
const_features = df.columns[df.nunique() <= 1]
display(const_features)

df = df.replace([np.inf, -np.inf], np.nan)
df = df.dropna()

df = df.drop_duplicates()

try:
    df = df.drop(columns=const_features)
    print(f"{const_features} and null values dropped.")
except:
    print(f"{const_features} already dropped.")

#display(df.columns)
display(df.shape)

### Redundant Features

Some features in this dataset do not contribute additional information for various reasons. For example, *fwd_header_length.1* is a known 'bug' of CICFlowMeter and is an exact duplicate of fwd_header_length. Other features, while not exact duplicates, are directly derived from existing variables and therefore contain highly redundant information. Since these values can be inferred from other features, explicitly including them may not provide additional predictive value and can unnecessarily increase the dimensionality of the dataset.e

##### Algebraic transformations
- *avg_fwd_segment_size* -> Exactly the same as *fwd_packet_length_mean* (same formula by a different name: **total_length_of_fwd_packets / total_fwd_packets**).
- *avg_bwd_segment_size* -> *bwd_packet_length_mean*.

##### Most connection flows consist of a single subflow
- *subflow_fwd_packets* -> *total_fwd_packets*.
- *subflow_fwd_bytes* -> *total_length_of_fwd_packets*.
- *subflow_bwd_packets* -> *total_backward_packets*.
- *subflow_bwd_bytes* -> *total_length_of_bwd_packets*.

##### Rate-Based Features
Features such as *flow_bytes/s*, *flow_packets/s*, *fwd_packets/s*, and *bwd_packets/s* are all derived from the general relationship:

**metric / flow_duration**

Given the underlying metric and the flow duration, these values can be directly computed. Therefore, they do not introduce additional information beyond what is already encoded in the original features.

In [ ]:
redundant_features = ['fwd_header_length.1', 'avg_fwd_segment_size', 'avg_bwd_segment_size', 'subflow_fwd_packets', 'subflow_fwd_bytes', 'subflow_bwd_packets', 
                      'subflow_bwd_bytes', 'flow_bytes/s', 'flow_packets/s', 'fwd_packets/s', 'bwd_packets/s']

try:
    df = df.drop(columns=redundant_features)
except:
    print("Redundant features already dropped.")

display(df.columns)
display(df.shape)

### Correlated Columns
Finally, a correlation analysis is performed to identify and remove highly correlated features from the dataset. A threshold of 0.95 is used; if any pair of features exhibits a correlation coefficient above this value, the corresponding variables are flagged for further analysis to determine whether one of them should be removed. This step helps reduce redundancy and mitigate multicollinearity in the feature space.

In [ ]:
threshold = 0.95

X = df.select_dtypes(include=["number"])

corr = X.corr(method="spearman").abs()
upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))

pairs = (
    upper.stack()
    .reset_index()
    .rename(columns={"level_0": "col1", "level_1": "col2", 0: "spearman"})
)

upper_pairs = pairs[pairs["spearman"] > threshold].sort_values("spearman", ascending=False)
pd.set_option("display.max_rows", None)
display(upper_pairs)

### Correlation Analysis
Based on the correlation matrix above and considering the nature of the data, four main groups of highly correlated features can be identified.

##### 1. Mathematic Correlation
- packet_length_variance = (packet_length_std)² -> The variance feature is redundant and will be removed.

##### 2. Packet Size
In traffic flood attacks, packet sizes tend to converge to similar values, causing several statistical descriptors to become highly redundant or flattened. In this context, only the most representative metrics are retained. We keep *fwd_packet_length_mean* and *bwd_packet_length_mean* and remove related features such as max, min, std, etc.

##### 3. IAT
The IAT (Inter-Arrival Time) is defined as the time between the arrival of consecutive packets. In automated traffic scenarios, this time can be highly regular, meaning that flow duration can often be approximated as: **flow_duration ≈ number_of_packets × average_IAT**

For this reason, we retain *flow_duration* and *fwd_iat_mean*/*bwd_iat_mean*. and remove the following highly correlated features: ['flow_iat_mean', 'flow_iat_total', 'fwd_iat_total', 'fwd_iat_std', 'fwd_iat_max', 'fwd_iat_min', 'bwd_iat_total', 'bwd_iat_std', 'bwd_iat_max', 'bwd_iat_min'].

##### 4. Active/Idle
As with IAT features, automated attack traffic can exhibit highly regular behavior, causing active and idle time metrics to become strongly correlated or nearly constant. These features are therefore reduced to their mean representations.

##### Special case: Flags
Due to protocol-level dependencies, *syn_flag_count* and *fwd_psh_flags* often represent nearly identical events. Similarly, *rst_flag_count* and *ece_flag_count* are highly correlated. In each pair, only one feature is retained.

In [ ]:
display(df[['fwd_psh_flags','syn_flag_count','rst_flag_count','ece_flag_count']].nunique())
display(df[['fwd_psh_flags','syn_flag_count','rst_flag_count','ece_flag_count']].value_counts())

In [ ]:
high_correlated_col1 = [col for col in upper_pairs['col1'] if any(upper_pairs["spearman"] > threshold)]
high_correlated_col2 = [col for col in upper_pairs['col2'] if any(upper_pairs["spearman"] > threshold)]
high_correlated = high_correlated_col1 + high_correlated_col2

persist_list = ['packet_length_std', 'fwd_packet_length_mean', 'bwd_packet_length_mean', 'flow_duration', 'fwd_iat_mean',
                'bwd_iat_mean', 'active_mean', 'idle_mean', 'syn_flag_count', 'rst_flag_count']

to_drop = list(set(high_correlated) - set(persist_list))
#missing = [c for c in persist if c not in df.columns]
#print("Columns in persist that don't exist:", missing)
#
df = df.drop(columns=to_drop)
display(to_drop)

In [ ]:
df.to_csv("./data/pre_processed/CIC-DDoS2019v2.csv", index=False)

display(df.shape)
display(df.columns)

In [ ]:
display(df.tail())